# 1. Export the decays of FLIM images based on a preselected ROI

## 1.1. Script 6d: Multi-ROI version with improved mask handling for time-series data

### 1.1.1 LIBRARY IMPORTS

In [1]:
# Import all required libraries for FLIM data processing
import tttrlib                      # Library for handling Time-Tagged Time-Resolved (TTTR) data
import pylab as plt                 # Matplotlib for plotting
import skimage as ski               # Image processing library
from skimage import io              # For reading/writing images
import numpy as np                  # Numerical computations
from mpl_toolkits.axes_grid1 import make_axes_locatable   # For plot formatting
import glob                         # File path pattern matching
import os                           # Operating system interface

### 1.1.2. FILE PATHS AND INSTRUMENT RESPONSE FUNCTION (IRF) SETUP

In [2]:
# Define the folder pattern to process - focuses on donor-only samples
# This version can switch between different sample types by commenting/uncommenting paths
path ='C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/*/*_ROI1*.tif'                  

# Load the Instrument Response Function (IRF) for deconvolution
# IRF represents the system's temporal response and is needed for lifetime analysis
filename_irf = 'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Calibration/IRF.ptu'
irf = tttrlib.TTTR(filename_irf)

### 1.1.3 CHANNEL AND TIME WINDOW CONFIGURATION

In [3]:
# Define detection channels for PIE (Pulsed Interleaved Excitation) setup
# PIE alternates between two excitation sources to separate donor and acceptor signals
green_ch_s = [0]           # Green channel, perpendicular polarization (s-polarized)
green_ch_p = [3]           # Green channel, parallel polarization (p-polarized)
red_ch_s = [1]             # Red channel, perpendicular polarization
red_ch_p = [2]             # Red channel, parallel polarization

# Time window configuration for PIE separation
# Total time window = 50 ns, TCSPC bin size = 10 ps → 5000 bins total
prompt_range = 0, 2500           # First half: donor excitation period (green laser on)
delay_range = 2500, 5000         # Second half: acceptor excitation period (red laser on)
dt = 0.02                        # Time resolution per bin in nanoseconds (20 ps)

### 1.1.4. IRF PROCESSING FOR EACH CHANNEL

In [4]:
# Extract and process IRF data for each detection channel
# This is used later for deconvolution and background correction

# Get micro times (arrival times within each excitation cycle)
micro_times = irf.micro_times

# Process green channel parallel polarization IRF
green_irf_p_indices = np.array(irf.get_selection_by_channel(green_ch_p))
green_irf_p = micro_times[green_irf_p_indices]

# Bin the data with 2-fold coarsening to match FLIM image binning
green_irf_p_counts = np.bincount(green_irf_p // 2)

# Process green channel perpendicular polarization IRF
green_irf_s_indices = np.array(irf.get_selection_by_channel(green_ch_s))
green_irf_s = micro_times[green_irf_s_indices]
green_irf_s_counts = np.bincount(green_irf_s // 2)

# Process red channel parallel polarization IRF
red_irf_p_indices = np.array(irf.get_selection_by_channel(red_ch_p))
red_irf_p = micro_times[red_irf_p_indices]
red_irf_p_counts = np.bincount(red_irf_p // 2)

# Process red channel perpendicular polarization IRF
red_irf_s_indices = np.array(irf.get_selection_by_channel(red_ch_s))
red_irf_s = micro_times[red_irf_s_indices]
red_irf_s_counts = np.bincount(red_irf_s // 2)

### 1.1.5. MAIN PROCESSING LOOP - ITERATE THROUGH ALL FILES

In [7]:
# Process each ROI file found in the specified path
for file in glob.glob(path):
    # Extract base filename without ROI suffix for finding corresponding .ptu file
    filename = os.path.abspath(file).split("_ROI")[0]
    
    # Load the TTTR data file (.ptu format from PicoQuant systems)
    data = tttrlib.TTTR(filename + '.ptu')
    
    # Load the ROI mask image (binary mask defining region of interest)
    mask = io.imread(file).astype(np.uint8)
    
    # Define output filename base for saving results
    save_as = os.path.abspath(file).split(".")[0]
    
    print('Processing....' + filename)
    
    # =============================================================================
    # 1.1.6. CREATE CLSM IMAGE OBJECTS FOR DIFFERENT CONDITIONS
    # =============================================================================
    # Create CLSM (Confocal Laser Scanning Microscopy) image containers
    # These separate the data by channel and time window
    
    # Green channel images (donor fluorescence)
    clsm_green_p = tttrlib.CLSMImage(tttr_data=data, channels=green_ch_p)
    clsm_green_s = tttrlib.CLSMImage(tttr_data=data, channels=green_ch_s)
    
    # Red channel images during prompt period (FRET-sensitized acceptor)
    clsm_red_prompt_p = tttrlib.CLSMImage(tttr_data=data, channels=red_ch_p, micro_time_ranges=[prompt_range])
    clsm_red_prompt_s = tttrlib.CLSMImage(tttr_data=data, channels=red_ch_s, micro_time_ranges=[prompt_range])
    
    # Red channel images during delay period (directly excited acceptor)
    clsm_red_delay_p = tttrlib.CLSMImage(tttr_data=data, channels=red_ch_p, micro_time_ranges=[delay_range])
    clsm_red_delay_s = tttrlib.CLSMImage(tttr_data=data, channels=red_ch_s, micro_time_ranges=[delay_range])

    # =============================================================================
    # 1.1.7. FILL IMAGE CONTAINERS WITH DATA
    # =============================================================================
    # Populate each CLSM image container with the appropriate photon data
    clsm_green_p.fill(data, channels=green_ch_p)  # green channels
    clsm_green_s.fill(data, channels=green_ch_s)  # green channels
    clsm_red_prompt_p.fill(data, channels=red_ch_p, micro_time_ranges=[prompt_range])  # red channels, prompt
    clsm_red_prompt_s.fill(data, channels=red_ch_s, micro_time_ranges=[prompt_range])  # red channels, prompt
    clsm_red_delay_s.fill(data, channels=red_ch_s, micro_time_ranges=[delay_range])  # red channels, delay
    clsm_red_delay_p.fill(data, channels=red_ch_p, micro_time_ranges=[delay_range])  # red channels, delay

    # =============================================================================
    # 1.1.8. IMPROVED MASK HANDLING FOR TIME-SERIES DATA
    # =============================================================================
    # Get image dimensions (frames, lines, pixels)
    n_frames, n_lines, n_pixel = clsm_green_p.shape
    
    def expand_mask(mask, n_frames):
        """
        Intelligently expand masks to match time-series data dimensions.
        
        Parameters:
        -----------
        mask : numpy.ndarray
            Input mask (2D or 3D)
        n_frames : int
            Number of frames in the time series
            
        Returns:
        --------
        numpy.ndarray
            3D mask with shape (n_frames, height, width)
        """
        if mask.ndim == 2:  # Single frame (2D)
            return np.tile(mask, (n_frames, 1, 1))  # Expand to match time series
        elif mask.shape[0] == n_frames:  # Already has the correct number of frames
            return mask
        else:
            raise ValueError(f"Unexpected mask dimensions: {mask.shape}")

    # Apply the smart mask expansion to handle both static ROIs and time-varying ROIs
    stack_mask = expand_mask(mask, n_frames)

    # =============================================================================
    # 1.1.9. GENERATE FLUORESCENCE DECAYS FROM ROI PIXELS
    # =============================================================================
    # Parameters for decay extraction
    kw_cell = {
        "tttr_data": data,
        "mask": stack_mask,    # ROI mask (now properly dimensioned)
        "tac_coarsening": 2,   # 2-fold binning: reduces to 2500 bins (20 ps/bin)
        "stack_frames": True   # Combine all frames in time series
    }
    
    # Extract fluorescence decays for each channel/condition
    # Green channels (donor fluorescence)
    decay_green_p = clsm_green_p.get_decay_of_pixels(**kw_cell)
    sum_decay_green_p = decay_green_p.sum(axis=0)   # Sum across all pixels in ROI
    decay_green_s = clsm_green_s.get_decay_of_pixels(**kw_cell)
    sum_decay_green_s = decay_green_s.sum(axis=0)
    
    # Red channels during prompt period (FRET-sensitized emission)
    decay_red_prompt_p = clsm_red_prompt_p.get_decay_of_pixels(**kw_cell)
    sum_decay_red_prompt_p = decay_red_prompt_p.sum(axis=0)
    decay_red_delay_p = clsm_red_delay_p.get_decay_of_pixels(**kw_cell)
    sum_decay_red_delay_p = decay_red_delay_p.sum(axis=0)
    
    # Red channels during delay period (directly excited acceptor)
    decay_red_prompt_s = clsm_red_prompt_s.get_decay_of_pixels(**kw_cell)
    sum_decay_red_prompt_s = decay_red_prompt_s.sum(axis=0)
    decay_red_delay_s = clsm_red_delay_s.get_decay_of_pixels(**kw_cell)
    sum_decay_red_delay_s = decay_red_delay_s.sum(axis=0)

    # =============================================================================
    # 1.1.10. EXPORT DECAY DATA IN JORDI FORMAT
    # =============================================================================
    # Save decay data in JORDI format (used by ChiSurf fitting software)
    # Format: parallel and perpendicular channels concatenated in single column
    
    # Green channel decay (donor, first 1250 bins only, i.e. 0 - 25 ns)
    jordi_counts_green = np.concatenate([sum_decay_green_p[0:1250], sum_decay_green_s[0:1250]])
    output_green = save_as + '_green.txt'
    np.savetxt(
        output_green, 
        np.vstack([jordi_counts_green]).T
        )
    
    # Red channel prompt decay (FRET-sensitized acceptor)
    jordi_counts_red_prompt = np.concatenate([sum_decay_red_prompt_p[0:1250], sum_decay_red_prompt_s[0:1250]])
    output_red = save_as + '_red_prompt.txt'
    np.savetxt(
        output_red, 
        np.vstack([jordi_counts_red_prompt]).T
        )
    
    # Red channel delay decay (directly excited acceptor, bins 1250-2500, i.e. 25- 50 ns)
    jordi_counts_red_delay = np.concatenate([sum_decay_red_delay_p[1250:2500], sum_decay_red_delay_s[1250:2500]])
    output_red = save_as + '_red_delay.txt'
    np.savetxt(
        output_red, 
        np.vstack([jordi_counts_red_delay]).T
        )

Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_64
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_64
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_64
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_64
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_64
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_64
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_66
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_66
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_66
Processing....C:\Users\Katherina Hemmen\Desktop\ExampleData\Raw_files\Cells\DA_1-5\low_FRET_1-5_66
Processing


## PART 2: EXPORT PHOTON STATISTICS AND BACKGROUND ANALYSIS


In [15]:
# =============================================================================
# 1.2.1 SETUP FOR STATISTICS EXPORT (DONOR-ONLY SAMPLES)
# =============================================================================
# Define paths for processing all files
path ='C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/DA*/*_ROI1*.tif'

# Output file path
save_file_as = 'C:/Users/Katherina Hemmen/Desktop/ExampleData/Raw_files/Cells/Photonnumbers_Multi.txt'

In [16]:
# =============================================================================
# 1.2.2 INITIALIZE STORAGE LISTS FOR STATISTICS
# =============================================================================
# Initialize lists to store parameters for all processed files
# These lists will grow as each file is processed
list_filenames = list()            # File names
list_bg_s_green = list()           # Green perpendicular background
list_bg_p_green = list()           # Green parallel background
list_nr_photons_s_green = list()   # Green perpendicular total photons
list_nr_photons_p_green = list()   # Green parallel total photons
list_bg_s_red_prompt = list()      # Red prompt perpendicular background
list_bg_p_red_prompt = list()      # Red prompt parallel background
list_nr_photons_s_red_prompt = list()  # Red prompt perpendicular total photons
list_nr_photons_p_red_prompt = list()  # Red prompt parallel total photons
list_bg_s_red_delay = list()       # Red delay perpendicular background
list_bg_p_red_delay = list()       # Red delay parallel background
list_nr_photons_s_red_delay = list()  # Red delay perpendicular total photons
list_nr_photons_p_red_delay = list()  # Red delay parallel total photons
list_nr_pixel_in_roi = list()      # Number of pixels in each ROI

In [17]:
# =============================================================================
# 1.2.3 PROCESS EACH FILE FOR PHOTON STATISTICS
# =============================================================================
for file in glob.glob(path):
    # Construct file paths for the exported decay data
    filename = os.path.abspath(file).split(".")[0]
    data_green = filename  + '_green.txt'              # Green decay file
    data_red_delay = filename + '_red_delay.txt'       # Red delay decay file
    data_red_prompt = filename + '_red_prompt.txt'     # Red prompt decay file
    ROI = file    # ROI mask file
    
    # Load decay data from text files (previously exported)
    decay_green = np.loadtxt(data_green, skiprows=0)            
    decay_red_delay = np.loadtxt(data_red_delay, skiprows=0)    
    decay_red_prompt = np.loadtxt(data_red_prompt, skiprows=0)
    
    # =============================================================================
    # 1.2.4. CALCULATE BACKGROUND LEVELS
    # =============================================================================
    # Background is estimated from early time bins (before main fluorescence peak)
    # Using bins 1-75 to avoid bin 0 artifacts and capture baseline level
    
    # Parallel channel backgrounds (first 1250 bins contain parallel data)
    bg_p_green = np.mean(decay_green[1:75])             # Green parallel background
    bg_p_red_delay = np.mean(decay_red_delay[1:75])     # Red delay parallel background  
    bg_p_red_prompt = np.mean(decay_red_prompt[1:75])   # Red prompt parallel background
    
    # Perpendicular channel backgrounds (bins 1251-2500 contain perpendicular data)
    bg_s_green = np.mean(decay_green[1251:1325])            # Green perpendicular background  
    bg_s_red_delay = np.mean(decay_red_delay[1251:1325])    # Red delay perpendicular background  
    bg_s_red_prompt = np.mean(decay_red_prompt[1251:1325])  # Red prompt perpendicular background
    
    # =============================================================================
    # 1.2.5 CALCULATE TOTAL PHOTON COUNTS
    # =============================================================================
    # Sum all photons in each decay curve (includes signal + background)
    
    # Green channel total counts
    sum_p_green = np.sum(decay_green[0:1250])     # Parallel polarization (bins 0-1249)
    sum_s_green = np.sum(decay_green[1250:2500])  # Perpendicular polarization (bins 1250-2499)
    
    # Red delay channel total counts (directly excited acceptor)
    sum_p_red_delay = np.sum(decay_red_delay[0:1250])      # Parallel
    sum_s_red_delay = np.sum(decay_red_delay[1250:2500])   # Perpendicular
    
    # Red prompt channel total counts (FRET-sensitized acceptor)
    sum_p_red_prompt = np.sum(decay_red_prompt[0:1250])      # Parallel
    sum_s_red_prompt = np.sum(decay_red_prompt[1250:2500])   # Perpendicular
    
    # =============================================================================
    # 1.2.6 COUNT ROI PIXELS
    # =============================================================================
    # Calculate number of pixels in the ROI mask
    ROI_img = io.imread(ROI)
    pixel_in_ROI = np.sum(ROI_img)  # Sum of binary mask = number of True pixels

    # =============================================================================
    # 1.2.7 STORE RESULTS IN LISTS
    # =============================================================================
    # Append all calculated parameters to their respective lists
    list_filenames.append(str(file))
    list_bg_p_green.append(bg_p_green)
    list_nr_photons_p_green.append(sum_p_green)
    list_bg_s_green.append(bg_s_green)
    list_nr_photons_s_green.append(sum_s_green)
    list_bg_s_red_prompt.append(bg_s_red_prompt)
    list_bg_p_red_prompt.append(bg_p_red_prompt)
    list_nr_photons_s_red_prompt.append(sum_s_red_prompt)
    list_nr_photons_p_red_prompt.append(sum_p_red_prompt)
    list_bg_s_red_delay.append(bg_s_red_delay)
    list_bg_p_red_delay.append(bg_p_red_delay)
    list_nr_photons_s_red_delay.append(sum_s_red_delay)
    list_nr_photons_p_red_delay.append(sum_p_red_delay)
    list_nr_pixel_in_roi.append(pixel_in_ROI)

# =============================================================================
# 1.2.8 EXPORT COMPREHENSIVE STATISTICS SUMMARY
# =============================================================================
# Define column headers for the output file
# Abbreviations: BG = Background, NrPhot = Number of Photons
# (p) = parallel, (s) = perpendicular, rp = red prompt, rd = red delay
header = 'Filename\tBGgreen(p)\tNrPhotgreen(p)\tBGgreen(s)\tNrPhotgreen(s)\tBGrp(p)\tNrPhotrp(p)\tBGrp(s)\tNrPhotrp(s)\tBGrd(p)\tNrPhotrd(p)\tBGrd(s)\tNrPhotrd(s)\tPixel in ROI'

# Save all statistics as a tab-delimited text file
# Each row represents one ROI file, columns contain different measurements
# This creates a comprehensive database for downstream analysis
np.savetxt(
    save_file_as,
    np.vstack(
        [
        list_filenames,                  # File identifiers
        list_bg_p_green,                 # Background measurements for offset removal
        list_nr_photons_p_green,         # Total photon counts for count rate calculation
        list_bg_s_green,
        list_nr_photons_s_green,
        list_bg_p_red_prompt,            # FRET-sensitized acceptor statistics
        list_nr_photons_p_red_prompt,
        list_bg_s_red_prompt,
        list_nr_photons_s_red_prompt,
        list_bg_p_red_delay,             # Direct excitation channel statistics
        list_nr_photons_p_red_delay,
        list_bg_s_red_delay,
        list_nr_photons_s_red_delay,
        list_nr_pixel_in_roi             # ROI size for normalization
        ],
    ).T,
    delimiter='\t', fmt="%s", header=header
)